# Distributed DP-SGD Training

This notebook demonstrates how to run DP-SGD training across multiple
GPUs using PyTorch's Distributed Data Parallel (DDP). The key
differences from single-device training are:

1. Each device computes clipped gradients on its local data shard.
2. Clipped gradients are summed across devices with `sum_gradients()`.
3. Noise is synchronized so all devices add the same noise.
4. `PoissonSampler` auto-detects distributed mode and shards data.

**Prerequisites:** [DP-SGD Training](dp_sgd_training.ipynb),
familiarity with `torchrun`.

**Components exercised:** `sum_gradients`, `sync_state`,
`PoissonSampler` in distributed mode, `gaussian_noise` with
synchronized RNG.

**Note:** This notebook is meant to be saved as a `.py` script and
launched with `torchrun --nproc_per_node=N`. The cells below show
the complete code with explanations.

## Setup and initialization

Initialize the distributed process group and assign each rank a
device.

In [ ]:
import os

import torch
import torch.distributed as dist
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

import opaque.accounting as acc
from opaque import PoissonSampler, clipped_grad, gaussian_noise, make_functional
from opaque.distributed import sum_gradients
from opaque.random import key

# Initialize distributed (set by torchrun)
dist.init_process_group(backend="nccl")
rank = dist.get_rank()
world_size = dist.get_world_size()
device = torch.device(f"cuda:{rank}")
torch.cuda.set_device(device)

print(f"Rank {rank}/{world_size} on {device}")

## Model and data

Every rank loads the same model. `PoissonSampler` automatically
detects the distributed environment and samples from disjoint
shards (SHARDED mode), so each rank sees different data.

In [ ]:
# Synthetic dataset (same on all ranks)
dataset_size = 10_000
X = torch.randn(dataset_size, 20)
y = (X[:, 0] + X[:, 1] > 0).float()
dataset = TensorDataset(X, y)


class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(20, 64)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        return self.fc2(F.relu(self.fc1(x))).squeeze(-1)


model = MLP().to(device)
fmodel, params = make_functional(model)

## DP components

Calibrate noise for the global training run. The sample rate is
based on the full dataset size (not per-rank).

`gaussian_noise` with `synchronized="auto"` (default) ensures all
ranks generate identical noise from the same RNG key.

In [ ]:
BATCH_SIZE = 64
NUM_STEPS = 500
TARGET_EPS = 3.0
DELTA = 1e-5
LR = 0.1

sample_rate = BATCH_SIZE / dataset_size

# Calibrate noise multiplier
result = acc.calibrate(
    acc.epsilon_budget(TARGET_EPS, delta=DELTA),
    lambda nm: acc.poisson(acc.gaussian(nm), sample_rate) * NUM_STEPS,
    param_min=0.1,
    param_max=10.0,
)
noise_multiplier = result.param

if rank == 0:
    print(f"Noise multiplier: {noise_multiplier:.4f}")

# Loss function
def loss_fn(params, x, y):
    logit = fmodel(params, x.unsqueeze(0)).squeeze()
    return F.binary_cross_entropy_with_logits(logit, y)

# Clipped gradient function
grad_fn, clip_state = clipped_grad(
    loss_fn, argnums=0, batch_argnums=(1, 2), l2_clip_norm=1.0,
)

# Noise function (synchronized across ranks)
noise_fn, noise_state = gaussian_noise(
    stddev=noise_multiplier * clip_state.sensitivity(),
    key=key(42),  # same key on all ranks => same noise
)

# Sampler: auto-detects distributed and uses SHARDED mode
sampler = PoissonSampler(
    dataset,
    sample_rate=sample_rate,
    num_epochs=NUM_STEPS,
    key=key(0),
)
loader = DataLoader(dataset, batch_sampler=sampler)

## Training loop

The key distributed operation is `sum_gradients()`, which
all-reduces the clipped gradients across ranks. After summing,
all ranks add the same synchronized noise.

In [ ]:
for step, (xb, yb) in enumerate(loader):
    if step >= NUM_STEPS:
        break

    xb, yb = xb.to(device), yb.to(device)

    # 1. Per-example clipped gradients (local data only)
    grads, clip_state = grad_fn(params, xb, yb, state=clip_state)

    # 2. Sum clipped gradients across all ranks
    grads = sum_gradients(grads)

    # 3. Add noise (identical on all ranks)
    noisy_grads, noise_state = noise_fn(grads, noise_state)

    # 4. SGD update
    bs = xb.shape[0] * world_size  # effective global batch size
    params = tuple(p - (LR / bs) * g for p, g in zip(params, noisy_grads))

    if rank == 0 and (step + 1) % 100 == 0:
        with torch.no_grad():
            loss = loss_fn(params, xb[0], yb[0]).item()
        eps = (
            acc.poisson(acc.gaussian(noise_multiplier), sample_rate)
            * (step + 1)
        ).epsilon_at(DELTA)
        print(f"Step {step+1:4d}  loss={loss:.4f}  epsilon={eps:.2f}")

## Cleanup

In [ ]:
if rank == 0:
    eps_final = (
        acc.poisson(acc.gaussian(noise_multiplier), sample_rate) * NUM_STEPS
    ).epsilon_at(DELTA)
    print(f"\nFinal epsilon: {eps_final:.2f}")

dist.destroy_process_group()

## Summary

Distributed DP-SGD with Opaque requires three changes from
single-device training:

| Step | Single device | Distributed |
|------|--------------|-------------|
| Gradient aggregation | (not needed) | `sum_gradients(grads)` |
| Noise synchronization | automatic | automatic (`synchronized="auto"`) |
| Data sampling | `PoissonSampler` | `PoissonSampler` (auto-shards) |

Launch with:

```bash
torchrun --nproc_per_node=4 distributed_dp_training.py
```

For adaptive clipping in distributed mode, use
`sync_adaptive_clip_state()` after each step to keep clip norms
consistent across ranks. See
[Distributed Training](../user-guide/distributed.md) in the user
guide.